Step 7 — Profitability, Operational Risk & Financial Impact Analysis

In [2]:
# Cell 1 — Load the dataset

import os
import numpy as np
import pandas as pd

PROJECT_DIR = r"nassau-candy-factory-optimization\data"

INPUT_FILE = os.path.join(
    PROJECT_DIR,
    "Nassau_Candy_Cleaned_Step.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "step7_outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())


Dataset loaded successfully.
Rows: 10194
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Shipping Lead Time Days', 'Financial Reconciliation Difference']


In [3]:
# Cell 2 — Validate financial and shipping columns
required_columns = [
    "Product ID",
    "Region",
    "Sales",
    "Gross Profit",
    "Cost",
    "Shipping Lead Time Days"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

print("Missing required columns:", missing_columns)

if missing_columns:
    raise ValueError(
        "The dataset is missing required columns: "
        + ", ".join(missing_columns)
    )

financial_columns = ["Sales", "Gross Profit", "Cost"]

for column in financial_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df["Shipping Lead Time Days"] = pd.to_numeric(
    df["Shipping Lead Time Days"],
    errors="coerce"
)

print("\nFinancial columns validated.")

Missing required columns: []

Financial columns validated.


In [5]:
# Cell 3 — Calculate historical profit margin

# calculate the historical gross profit margin as:

# $$ \text{Gross Profit Margin (\%)} = \frac{\text{Gross Profit}}{\text{Sales}} \times 100 $$

# This calculation is meaningful when sales are nonzero and the financial fields use consistent accounting definitions.

df["Gross Profit Margin (%)"] = np.where(
    df["Sales"].notna() & (df["Sales"] != 0),
    (df["Gross Profit"] / df["Sales"]) * 100,
    np.nan
)

print("Profit margin calculated.")

display(
    df[
        [
            "Product ID",
            "Sales",
            "Gross Profit",
            "Cost",
            "Gross Profit Margin (%)"
        ]
    ].head(10)
)

# Interpretation:

# A positive margin indicates positive gross profit relative to sales.
# A negative margin indicates negative gross profit relative to sales, assuming the source fields are consistent.
# A missing margin may indicate missing financial data or zero sales.

# Do not assume that gross profit equals sales minus cost until you verify how the original dataset defines these fields.


Profit margin calculated.


,Product ID,Sales,Gross Profit,Cost,Gross Profit Margin (%)
0,CHO-MIL-31000,6.50,4.22,2.28,64.923077
1,CHO-TRI-54000,7.50,4.90,2.60,65.333333
2,CHO-NUT-13000,10.47,7.47,3.00,71.346705
3,CHO-SCR-58000,10.80,7.50,3.30,69.444444
4,CHO-TRI-54000,11.25,7.35,3.90,65.333333
5,CHO-SCR-58000,32.40,22.50,9.90,69.444444
6,CHO-TRI-54000,7.50,4.90,2.60,65.333333
7,CHO-MIL-31000,9.75,6.33,3.42,64.923077
8,CHO-NUT-13000,6.98,4.98,2.00,71.346705
9,CHO-MIL-31000,13.00,8.44,4.56,64.923077


In [6]:
# Cell 4 — Create an overall financial summary
financial_summary = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Gross Profit",
        "Total Cost",
        "Average Gross Profit Margin (%)",
        "Records with Missing Sales",
        "Records with Missing Gross Profit",
        "Records with Missing Cost"
    ],
    "Value": [
        df["Sales"].sum(min_count=1),
        df["Gross Profit"].sum(min_count=1),
        df["Cost"].sum(min_count=1),
        df["Gross Profit Margin (%)"].mean(),
        df["Sales"].isna().sum(),
        df["Gross Profit"].isna().sum(),
        df["Cost"].isna().sum()
    ]
})

display(financial_summary)

financial_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "overall_financial_summary.csv"
    ),
    index=False
)


,Metric,Value
0,Total Sales,141783.630000
1,Total Gross Profit,93442.800000
2,Total Cost,48340.830000
3,Average Gross Profit Margin (%),66.514045
4,Records with Missing Sales,0.000000
5,Records with Missing Gross Profit,0.000000
6,Records with Missing Cost,0.000000


In [7]:
# Cell 5 — Calculate the overall weighted gross profit margin
total_sales = df["Sales"].sum(min_count=1)
total_gross_profit = df["Gross Profit"].sum(min_count=1)

if pd.notna(total_sales) and total_sales != 0:
    weighted_gross_profit_margin = (
        total_gross_profit / total_sales
    ) * 100
else:
    weighted_gross_profit_margin = np.nan

print(
    "Overall weighted gross profit margin:",
    (
        f"{weighted_gross_profit_margin:.2f}%"
        if pd.notna(weighted_gross_profit_margin)
        else "Not calculable"
    )
)


Overall weighted gross profit margin: 65.91%


In [8]:
#Cell 6 — Analyze profitability by product
product_financial_summary = (
    df.groupby("Product ID", dropna=False)
    .agg(
        Order_Lines=("Product ID", "size"),
        Total_Sales=("Sales", "sum"),
        Total_Gross_Profit=("Gross Profit", "sum"),
        Total_Cost=("Cost", "sum"),
        Average_Sales=("Sales", "mean"),
        Average_Gross_Profit=("Gross Profit", "mean"),
        Average_Shipping_Lead_Time=("Shipping Lead Time Days", "mean"),
        Shipping_Lead_Time_Std=("Shipping Lead Time Days", "std")
    )
    .reset_index()
)

product_financial_summary["Gross Profit Margin (%)"] = np.where(
    product_financial_summary["Total_Sales"] != 0,
    (
        product_financial_summary["Total_Gross_Profit"]
        / product_financial_summary["Total_Sales"]
    ) * 100,
    np.nan
)

product_financial_summary = product_financial_summary.sort_values(
    "Total_Gross_Profit",
    ascending=False
)

display(product_financial_summary.head(15))

product_financial_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "product_financial_risk_summary.csv"
    ),
    index=False
)


#For each product, it calculates:

#Total sales
#Total gross profit
#Total recorded cost
#Average gross profit
#Average shipping lead time
#Standard deviation of shipping lead time
#Gross profit margin

# Important: A product with high historical gross profit is not automatically a good candidate for reassignment. We must also consider service performance, feasible factories, incremental costs, and the uncertainty of any predicted improvement.


,Product ID,Order_Lines,Total_Sales,Total_Gross_Profit,Total_Cost,Average_Sales,Average_Gross_Profit,Average_Shipping_Lead_Time,Shipping_Lead_Time_Std,Gross Profit Margin (%)
3,CHO-SCR-58000,2064,27874.80,19357.50,8517.30,13.505233,9.378634,1320.641957,260.121265,69.444444
4,CHO-TRI-54000,2015,28485.00,18610.20,9874.80,14.136476,9.235831,1325.661042,260.524339,65.333333
1,CHO-MIL-31000,2137,26867.75,17443.37,9424.38,12.572649,8.162550,1316.764623,263.692704,64.923077
2,CHO-NUT-13000,1810,23574.95,16819.95,6755.00,13.024834,9.292790,1328.915470,260.837398,71.346705
0,CHO-FUD-51000,1818,24890.40,16593.60,8296.80,13.691089,9.127393,1314.242024,264.102825,66.666667
8,OTH-LIC-15000,94,7860.00,3930.00,3930.00,83.617021,41.808511,1339.138298,288.459855,50.000000
6,OTH-GUM-21000,120,597.50,310.70,286.80,4.979167,2.589167,1306.525000,292.031596,52.000000
9,SUG-EVE-47000,3,130.00,104.00,26.00,43.333333,34.666667,1394.666667,213.336198,80.000000
7,OTH-KAZ-38000,96,1205.75,92.75,1113.00,12.559896,0.966146,1272.989583,259.943212,7.692308
11,SUG-HAI-55000,4,76.50,59.50,17.00,19.125000,14.875000,1455.250000,209.876750,77.777778


In [9]:
#Cell 7 — Analyze profitability by region

region_financial_summary = (
    df.groupby("Region", dropna=False)
    .agg(
        Order_Lines=("Region", "size"),
        Total_Sales=("Sales", "sum"),
        Total_Gross_Profit=("Gross Profit", "sum"),
        Total_Cost=("Cost", "sum"),
        Average_Lead_Time=("Shipping Lead Time Days", "mean"),
        Median_Lead_Time=("Shipping Lead Time Days", "median"),
        Lead_Time_Std=("Shipping Lead Time Days", "std")
    )
    .reset_index()
)

region_financial_summary["Gross Profit Margin (%)"] = np.where(
    region_financial_summary["Total_Sales"] != 0,
    (
        region_financial_summary["Total_Gross_Profit"]
        / region_financial_summary["Total_Sales"]
    ) * 100,
    np.nan
)

region_financial_summary = region_financial_summary.sort_values(
    "Total_Gross_Profit",
    ascending=False
)

display(region_financial_summary)

region_financial_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "region_financial_risk_summary.csv"
    ),
    index=False
)



,Region,Order_Lines,Total_Sales,Total_Gross_Profit,Total_Cost,Average_Lead_Time,Median_Lead_Time,Lead_Time_Std,Gross Profit Margin (%)
3,Pacific,3253,46301.53,30485.94,15815.59,1322.194897,1274.0,266.470647,65.842187
0,Atlantic,2986,41197.24,26973.70,14223.54,1322.745144,1274.0,256.541810,65.474532
2,Interior,2335,32037.60,21282.49,10755.11,1323.091221,1274.0,262.693597,66.429726
1,Gulf,1620,22247.26,14700.67,7546.59,1311.374691,1274.0,264.727855,66.078564


In [11]:
# Cell 8 — Measure operational shipping risk

# A higher standard deviation indicates that recorded lead times vary more. It does not, on its own, prove the presence of congestion, supply disruption, or factory inefficiency.

valid_shipping = df.loc[
    df["Shipping Lead Time Days"].notna()
    & (df["Shipping Lead Time Days"] >= 0)
].copy()

shipping_risk_summary = (
    valid_shipping.groupby(
        ["Region", "Ship Mode"],
        dropna=False
    )
    .agg(
        Shipment_Records=("Shipping Lead Time Days", "count"),
        Average_Lead_Time=("Shipping Lead Time Days", "mean"),
        Median_Lead_Time=("Shipping Lead Time Days", "median"),
        Lead_Time_Std=("Shipping Lead Time Days", "std"),
        Minimum_Lead_Time=("Shipping Lead Time Days", "min"),
        Maximum_Lead_Time=("Shipping Lead Time Days", "max")
    )
    .reset_index()
)

shipping_risk_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "shipping_operational_risk.csv"
    ),
    index=False
)

display(
    shipping_risk_summary.sort_values(
        "Lead_Time_Std",
        ascending=False,
        na_position="last"
    ).head(15)
)
# INTERPRETATION: 
#Average lead time: Typical recorded duration, summarized using the mean.
#Median lead time: Middle recorded duration.
#Lead-time standard deviation: Measure of variability.
#Maximum lead time: Largest recorded duration in the group.

#Groups with consistently high lead times or substantial variability may warrant investigation. Before drawing conclusions, check shipment counts and whether unusual observations distort the results.

,Region,Ship Mode,Shipment_Records,Average_Lead_Time,Median_Lead_Time,Lead_Time_Std,Minimum_Lead_Time,Maximum_Lead_Time
8,Interior,First Class,303,1352.435644,1272.0,279.076626,905.0,1638.0
6,Gulf,Second Class,329,1324.717325,1273.0,275.555330,906.0,1640.0
4,Gulf,First Class,234,1319.824786,1272.0,273.817393,905.0,1638.0
12,Pacific,First Class,515,1340.935922,1272.0,267.518016,905.0,1638.0
15,Pacific,Standard Class,1916,1313.231733,1274.0,266.516604,908.0,1642.0
14,Pacific,Second Class,637,1327.543171,1273.0,266.197815,906.0,1640.0
11,Interior,Standard Class,1447,1314.657913,1274.0,262.958250,908.0,1642.0
13,Pacific,Same Day,185,1344.437838,1269.0,261.597595,904.0,1636.0
7,Gulf,Standard Class,974,1306.927105,1274.0,260.226749,908.0,1642.0
3,Atlantic,Standard Class,1783,1319.303982,1274.0,258.758821,908.0,1642.0


In [12]:
# Cell 9 — Calculate profit-impact stability

# A factory reassignment's actual profit impact cannot be calculated without a defensible estimate of the incremental revenue and costs associated with the change.

#For this initial version, we will use the coefficient of variation of recorded gross profit:

# $$ CV_{\text{profit}} = \frac{\text{Standard Deviation of Gross Profit}} {|\text{Mean Gross Profit}|} $$

#This indicator is useful when there are enough observations and the mean gross profit is not near zero.

profit_stability = (
    df.groupby("Product ID", dropna=False)
    .agg(
        Record_Count=("Gross Profit", "count"),
        Mean_Gross_Profit=("Gross Profit", "mean"),
        Std_Gross_Profit=("Gross Profit", "std")
    )
    .reset_index()
)

profit_stability["Gross Profit CV"] = np.where(
    (
        profit_stability["Mean_Gross_Profit"].notna()
        & (profit_stability["Mean_Gross_Profit"].abs() > 1e-9)
        & (profit_stability["Record_Count"] >= 2)
    ),
    (
        profit_stability["Std_Gross_Profit"]
        / profit_stability["Mean_Gross_Profit"].abs()
    ),
    np.nan
)

def stability_category(row):
    if row["Record_Count"] < 2:
        return "Insufficient observations"

    if pd.isna(row["Gross Profit CV"]):
        return "Not reliably calculable"

    if row["Gross Profit CV"] <= 0.25:
        return "Lower historical variability"

    if row["Gross Profit CV"] <= 0.50:
        return "Moderate historical variability"

    return "Higher historical variability"

profit_stability["Historical Profit Stability"] = (
    profit_stability.apply(stability_category, axis=1)
)

display(profit_stability.head(15))

profit_stability.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "historical_profit_stability.csv"
    ),
    index=False
)
#Important interpretation

#The categories use transparent thresholds for exploratory analysis:

#CV up to 0.25: lower historical variability
#CV above 0.25 and up to 0.50: moderate historical variability
#CV above 0.50: higher historical variability



,Product ID,Record_Count,Mean_Gross_Profit,Std_Gross_Profit,Gross Profit CV,Historical Profit Stability
0,CHO-FUD-51000,1818,9.127393,5.273759,0.577795,Higher historical variability
1,CHO-MIL-31000,2137,8.162550,4.859998,0.595402,Higher historical variability
2,CHO-NUT-13000,1810,9.292790,5.404772,0.581609,Higher historical variability
3,CHO-SCR-58000,2064,9.378634,5.565864,0.593462,Higher historical variability
4,CHO-TRI-54000,2015,9.235831,5.423458,0.587219,Higher historical variability
5,OTH-FIZ-56000,6,7.875000,2.359820,0.299660,Moderate historical variability
6,OTH-GUM-21000,120,2.589167,1.685295,0.650902,Higher historical variability
7,OTH-KAZ-38000,96,0.966146,0.511216,0.529129,Higher historical variability
8,OTH-LIC-15000,94,41.808511,24.272767,0.580570,Higher historical variability
9,SUG-EVE-47000,3,34.666667,12.220202,0.352506,Moderate historical variability


In [13]:
#Cell 10 — Create a financial and operational risk report
risk_report = product_financial_summary[
    [
        "Product ID",
        "Order_Lines",
        "Total_Sales",
        "Total_Gross_Profit",
        "Gross Profit Margin (%)",
        "Average_Shipping_Lead_Time",
        "Shipping_Lead_Time_Std"
    ]
].merge(
    profit_stability[
        [
            "Product ID",
            "Gross Profit CV",
            "Historical Profit Stability"
        ]
    ],
    on="Product ID",
    how="left"
)

risk_report["Factory Reallocation Financial Impact"] = (
    "Not measurable: validated factory assignment and "
    "incremental scenario costs are unavailable"
)

risk_report["Reallocation Recommendation Status"] = (
    "Pending factory-aware lead-time and financial analysis"
)

risk_report.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step7_financial_operational_risk_report.csv"
    ),
    index=False
)

display(risk_report.head(15))

# This brings the historical profitability and operational risk indicators together while preventing the analysis from presenting unsupported financial projections.


,Product ID,Order_Lines,Total_Sales,Total_Gross_Profit,Gross Profit Margin (%),Average_Shipping_Lead_Time,Shipping_Lead_Time_Std,Gross Profit CV,Historical Profit Stability,Factory Reallocation Financial Impact,Reallocation Recommendation Status
0,CHO-SCR-58000,2064,27874.80,19357.50,69.444444,1320.641957,260.121265,0.593462,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
1,CHO-TRI-54000,2015,28485.00,18610.20,65.333333,1325.661042,260.524339,0.587219,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
2,CHO-MIL-31000,2137,26867.75,17443.37,64.923077,1316.764623,263.692704,0.595402,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
3,CHO-NUT-13000,1810,23574.95,16819.95,71.346705,1328.915470,260.837398,0.581609,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
4,CHO-FUD-51000,1818,24890.40,16593.60,66.666667,1314.242024,264.102825,0.577795,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
5,OTH-LIC-15000,94,7860.00,3930.00,50.000000,1339.138298,288.459855,0.580570,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
6,OTH-GUM-21000,120,597.50,310.70,52.000000,1306.525000,292.031596,0.650902,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
7,SUG-EVE-47000,3,130.00,104.00,80.000000,1394.666667,213.336198,0.352506,Moderate historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
8,OTH-KAZ-38000,96,1205.75,92.75,7.692308,1272.989583,259.943212,0.529129,Higher historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...
9,SUG-HAI-55000,4,76.50,59.50,77.777778,1455.250000,209.876750,0.352941,Moderate historical variability,Not measurable: validated factory assignment a...,Pending factory-aware lead-time and financial ...


In [14]:
# Cell 11 — Export a Step 7 summary
summary_path = os.path.join(
    OUTPUT_DIR,
    "step7_summary.txt"
)

with open(summary_path, "w", encoding="utf-8") as file:
    file.write(
        "NASSAU CANDY DISTRIBUTOR - STEP 7\n"
        "PROFITABILITY, OPERATIONAL RISK, AND FINANCIAL IMPACT\n\n"
        "Historical financial metrics are calculated from the supplied data.\n"
        "Shipping variability is used as an exploratory risk indicator.\n"
        "Profit stability refers to historical gross-profit variability.\n\n"
        "Limitations:\n"
        "- Current product-to-factory assignments have not been validated.\n"
        "- The baseline shipping model is not factory-aware.\n"
        "- Incremental factory transportation and manufacturing costs\n"
        "  are not supported by the currently identified data.\n"
        "- Historical gross profit is not incremental scenario profit.\n"
        "- Factory-specific reallocation recommendations remain pending.\n"
    )

print("Step 7 completed.")
print("Output folder:", OUTPUT_DIR)
print("\nFiles created:")

for filename in os.listdir(OUTPUT_DIR):
    print("-", filename)

Step 7 completed.
Output folder: nassau-candy-factory-optimization\data\step7_outputs

Files created:
- historical_profit_stability.csv
- overall_financial_summary.csv
- product_financial_risk_summary.csv
- region_financial_risk_summary.csv
- shipping_operational_risk.csv
- step7_financial_operational_risk_report.csv
- step7_summary.txt
